# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [2]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [4]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [5]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [6]:
# Make sure the date columns are proper dates
for col in ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']:
    df[col] = pd.to_datetime(df[col])

# Reference date: the data is a snapshot from around January 2016
ref_date = pd.Timestamp('2016-01-01')

df['tenure_years'] = (ref_date - df['date_activ']).dt.days // 365
df['months_to_end'] = (df['date_end'] - ref_date).dt.days // 30
df['months_since_modif'] = (ref_date - df['date_modif_prod']).dt.days // 30
df['months_to_renewal'] = (df['date_renewal'] - ref_date).dt.days // 30

df[['tenure_years', 'months_to_end', 'months_since_modif', 'months_to_renewal']].describe()

,tenure_years,months_to_end,months_since_modif,months_to_renewal
count,14606.000000,14606.000000,14606.000000,14606.000000
mean,4.429276,6.506025,35.988909,-5.935848
std,1.655558,3.550940,30.724066,3.956654
min,1.000000,0.000000,-1.000000,-31.000000
25%,3.000000,3.000000,6.000000,-9.000000
50%,4.000000,7.000000,30.000000,-6.000000
75%,5.000000,10.000000,65.000000,-3.000000
max,12.000000,17.000000,154.000000,0.000000


**New features (time-based):**
- **tenure_years** – how long the customer has been with PowerCo. Long-standing customers may be more loyal.
- **months_to_end** – months until the contract ends. Customers are most likely to switch when their contract is about to expire.
- **months_since_modif** – months since the product was last changed. Recent changes (e.g. a price or plan change) may trigger churn.
- **months_since_renewal** – months since the contract was last renewed. Originally calculated as months *to* renewal, but every value was ≤ 0, showing all renewal dates fall before the reference date, so it was reversed.

In [7]:
df['months_since_renewal'] = (ref_date - df['date_renewal']).dt.days // 30
df = df.drop(columns=['months_to_renewal'])
df['months_since_renewal'].describe()

count    14606.00000
mean         4.96789
std          3.94460
min         -1.00000
25%          2.00000
50%          5.00000
75%          8.00000
max         30.00000
Name: months_since_renewal, dtype: float64

In [8]:
price_df = pd.read_csv('price_data.csv')
price_df['price_date'] = pd.to_datetime(price_df['price_date'])
price_df = price_df.sort_values(['id', 'price_date'])

# Month-to-month change in off-peak energy price for each customer
price_df['offpeak_change'] = price_df.groupby('id')['price_off_peak_var'].diff()

price_features = price_df.groupby('id').agg(
    max_offpeak_increase=('offpeak_change', 'max'),
    offpeak_price_volatility=('price_off_peak_var', 'std'),
    mean_offpeak_price=('price_off_peak_var', 'mean')
).reset_index()

df = df.merge(price_features, on='id', how='left')
df[['max_offpeak_increase', 'offpeak_price_volatility', 'mean_offpeak_price']].describe()

,max_offpeak_increase,offpeak_price_volatility,mean_offpeak_price
count,14606.000000,14606.000000,14606.000000
mean,0.003560,0.004069,0.142327
std,0.012645,0.004970,0.022512
min,0.000000,0.000000,0.000000
25%,0.000000,0.002153,0.124430
50%,0.000670,0.002987,0.147630
75%,0.001644,0.004243,0.150415
max,0.235160,0.068978,0.278098


**New features (price-based):**
- **max_offpeak_increase** – the largest single month-to-month rise in a customer's off-peak energy price. A sudden price jump may push a customer to switch, even if the yearly change is small.
- **offpeak_price_volatility** – how much the customer's off-peak price fluctuated over the year (standard deviation). Unstable pricing may reduce trust.
- **mean_offpeak_price** – the customer's average off-peak price, capturing how expensive their tariff is overall.

These extend Estelle's December–January feature by capturing price behaviour across the whole year, directly testing the price-sensitivity hypothesis.

A small number of customers have a mean price of 0, which may indicate missing price data.”

In [9]:
import numpy as np

# 1. has_gas: 't'/'f' -> 1/0
df['has_gas'] = df['has_gas'].replace({'t': 1, 'f': 0}).astype(int)

# 2. One-hot encode the categorical columns (MISSING becomes its own category)
df = pd.get_dummies(df, columns=['channel_sales', 'origin_up'], dtype=int)

# 3. Log-transform the heavily skewed consumption columns
for col in ['cons_12m', 'cons_gas_12m', 'cons_last_month', 'forecast_cons_12m']:
    df[col] = np.log1p(df[col])

# 4. Drop raw date columns (replaced by the duration features)
df = df.drop(columns=['date_activ', 'date_end', 'date_modif_prod', 'date_renewal'])

df.shape

(14606, 59)

**Preparing features for modelling:**
- **has_gas** converted from 't'/'f' to 1/0.
- **channel_sales** and **origin_up** one-hot encoded, so each category becomes its own 0/1 column. MISSING is kept as its own category, since EDA showed it has a distinct churn rate.
- **Consumption columns** log-transformed (log1p) to reduce the heavy right skew found in EDA. log1p is used because many values are 0.
- **Raw date columns** dropped, as they are replaced by the duration features above.

In [10]:
# Remove: near-identical columns add nothing new
print(df[['margin_gross_pow_ele', 'margin_net_pow_ele']].corr())

# Combine: price gap between peak and off-peak energy
df['peak_offpeak_spread'] = df['forecast_price_energy_peak'] - df['forecast_price_energy_off_peak']

df = df.drop(columns=['margin_gross_pow_ele'])

                      margin_gross_pow_ele  margin_net_pow_ele
margin_gross_pow_ele              1.000000            0.999914
margin_net_pow_ele                0.999914            1.000000


**Removing and combining features:**
- margin_gross_pow_ele and margin_net_pow_ele were almost perfectly correlated, so the gross margin column was dropped as it adds no new information.
- **peak_offpeak_spread** combines the forecast peak and off-peak energy prices. A large gap between peak and off-peak pricing may make a tariff feel unfair or expensive at busy times, which could drive churn.

**Summary:** Created time-based features (tenure, months to contract end, months since modification and renewal), price-based features (largest monthly off-peak increase, price volatility, average price) and a combined peak/off-peak price spread to extend Estelle's December–January feature. A near-duplicate margin column was removed, categorical columns were encoded, skewed consumption columns log-transformed and raw date columns dropped, leaving a fully numeric dataset ready for modelling.